# ANLP Assignment 2: Mixture of Experts, Pretraining Optimizers, and Decoding Strategies
### Complete Self-Contained GPU Runner for Google Colab Pro (T4 / V100 / A100)

Welcome! This notebook executes all three core sections of **Assignment 2 (Monsoon 2026, IIIT Hyderabad)**:
1. **Part 1: Mixture of Experts (MoE) (40 marks)** — 5 FFN variants, parameter parity matching, Switch auxiliary loss, and language specialization heatmaps on `belumind/en-vi-ja-curated-500k-triplets`.
2. **Part 2: Custom Optimizers from Scratch (30 marks)** — Category 1 (AdamW), Category 2 (Adan), Category 3 (Lion), Category 4 (Muon), and Bonus (Sophia) pretraining dynamics on `browndw/human-ai-parallel-corpus`.
3. **Part 3: Decoding Strategies from Scratch (30 marks)** — Greedy, Top-k, Top-p, and Beam Search ($B \in \{1, 2, 4\}$) on `EleutherAI/Pythia-160M` evaluated on `hamishivi/ROCStories` (1,000 samples) using strictly `model.forward()`.

---
### 👶 Explain Like I'm 5 (ELI5): What Are We Doing in This Notebook?
Imagine you are building a team of smart robots to read and write stories:
- **Part 1 (Mixture of Experts)**: Instead of asking one big robot to know every language, we have a smart receptionist that sends Vietnamese words to one expert robot and Japanese words to another expert robot! They share the work so each robot does what it's best at.
- **Part 2 (Optimizers)**: How do robots learn from their mistakes? An optimizer is like a coach telling the robot how big of a step to take. We test 5 different coaches from scratch (like Muon, which uses matrix geometry, and Lion, which saves half the memory) to see who trains the robot fastest!
- **Part 3 (Decoding Strategies)**: When the robot writes the next word of a story, how does it choose? Greedy always takes the highest score; Top-k and Top-p pick from the best fun choices so the story isn't boring; Beam search looks ahead like a chess master to find the best storyline!

---
### ⏱️ Expected Runtimes on Google Colab Pro
| Pipeline Stage | Fast Verification / Smoke Test | Full Dataset Training (Colab Pro A100/V100) |
| :--- | :--- | :--- |
| **Environment Setup & Verification** | ~1 minute | ~1 minute |
| **Part 1: 5 MoE Ablations** | ~2 minutes (dummy tensors) | ~45-60 minutes (A100) / ~90 minutes (T4) |
| **Part 2: 5 Optimizers Dynamics** | ~1 minute (warmup) | ~30-40 minutes (A100) / ~60 minutes (T4) |
| **Part 3: Decoding Benchmark (1,000 samples)** | ~30 seconds (10 samples) | ~10-15 minutes (1,000 samples on A100) |
| **Total End-to-End Execution** | **~5 minutes** | **~1.5 to 2.5 hours** |

---
### 📂 Where Are Files & Checkpoints Generated?
- Trained Model Checkpoints: `checkpoints/part1/` and `checkpoints/part2/`
- Heatmaps & Plots: `assets/expert_heatmap.png`, `assets/optimizer_dynamics.png`, `assets/beam_search_runtime.png`
- Numerical Results & Logs: `assets/part2/optimizer_benchmark_results.json`, `assets/part3/decoding_summary.json`
- At the end of this notebook, a single cell zips all outputs into `submission_artifacts.zip` for instant local download.

## Step 1: GPU Diagnostics & Setup (ELI5: Check that the supercomputer is awake)

In [ ]:
# 1. Verify GPU assignment
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Allocated GPU: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Install Libraries (ELI5: Download the tools our robot needs)

In [ ]:
# Install core NLP packages
!pip install -q torch transformers datasets tokenizers sacrebleu evaluate wandb matplotlib seaborn einops

## Step 3: Clone Codebase or Mount Drive (ELI5: Put our project on the computer)

In [ ]:
import os, sys
# If you have pushed to your GitHub repository ANLP_assignment2, you can clone it directly:
# !git clone https://github.com/<your-username>/ANLP_assignment2.git
# %cd ANLP_assignment2

# Ensure repo root is on python path
sys.path.append(os.getcwd())
print(f"Working Directory: {os.getcwd()}")
print(f"Files in workspace: {os.listdir('.')[:10]}")

## Step 4: Verify Mathematical Integrity (ELI5: Double check all formulas and code tests)

In [ ]:
# Runs comprehensive pytest suite across all 22 mathematical tests and main verify
!python -m pytest tests/ -v
!python main.py verify

## Step 5: Part 1 — Mixture of Experts (MoE) Execution
### ELI5:
We test 5 designs:
1. **Dense**: 1 big standard feedforward brain.
2. **Top-1 MoE**: 4 experts, 1 chosen per token.
3. **Top-2 MoE**: 4 experts, 2 chosen per token.
4. **Shared + Routed**: 1 expert always stays on, plus 1 routed expert chosen from 3.
5. **Matched Active**: 4 experts with doubled width so active parameters match the dense baseline exactly.

We train on `belumind/en-vi-ja-curated-500k-triplets` to translate Vietnamese & Japanese to English, track routing with Switch auxiliary loss, and extract language heatmaps.

In [ ]:
# Run Part 1 training across all 5 architectural ablations
# To run a quick smoke run, keep samples=2000, epochs=1.
# For full training, increase samples=10000, epochs=3.

ablations = ["dense", "top1", "top2", "shared_routed", "matched_active"]
for abl in ablations:
    print(f"\n{'='*25} RUNNING PART 1 ABLATION: {abl.upper()} {'='*25}")
    !python main.py part1 --ablation {abl} --train --samples 2000 --epochs 1

## Step 6: Part 2 — Custom Optimizers Pretraining Dynamics
### ELI5:
We pretrain our dense model on `browndw/human-ai-parallel-corpus` using 5 scratch optimizers:
1. **AdamW**: Standard baseline with decoupled weight decay.
2. **Adan**: Looks at gradient differences to smooth out bumpy steps.
3. **Lion**: Uses only `+1` or `-1` signs, cutting memory in half!
4. **Muon**: Uses Newton-Schulz polynomial iterations to rotate matrices without distortion.
5. **Sophia**: Uses second-order diagonal curvature and clipping to handle steep hills.

We log validation loss and test BLEU at $0.1\times$ intervals to replicate Figure 2 of *Fantastic Pretraining Optimizers*.

In [ ]:
# Run Part 2 Pretraining Dynamics benchmark for all 5 optimizers
# tokens=100000 provides quick validation; tokens=500000 gives full Chinchilla curve.
!python main.py part2 --optimizer all --train --tokens 100000

## Step 7: Part 3 — Decoding Strategies on ROCStories
### ELI5:
Using `EleutherAI/Pythia-160M` without calling `model.generate()`, we generate story endings for 1,000 ROCStories samples using:
- **Greedy**: Always pick top token.
- **Top-k ($k=40, T=0.8$)**: Pick from top 40 words.
- **Top-p ($p=0.9, T=0.8$)**: Pick from the top 90% probability mass.
- **Beam Search ($B=1, 2, 4$)**: Keep the best $B$ candidate story paths.

We measure perplexity, accuracy, and confirm linear latency scaling with respect to beam width $B$.

In [ ]:
# Run Part 3 Decoding Benchmark on 1000 samples
!python main.py part3 --model_name EleutherAI/Pythia-160M --samples 1000 --output_dir assets/part3

## Step 8: View Visualizations (ELI5: Look at the pretty pictures our robots made!)

In [ ]:
from IPython.display import Image, display

figs = [
    ("MoE Language Specialization Heatmap", "assets/expert_heatmap.png"),
    ("Optimizer Pretraining Dynamics (Fig 2)", "assets/optimizer_dynamics.png"),
    ("Beam Search Latency vs Beam Width", "assets/beam_search_runtime.png")
]

for title, path in figs:
    if os.path.exists(path):
        print(f"\n=== {title} ({path}) ===")
        display(Image(path))
    else:
        print(f"Note: {path} not yet generated.")

## Step 9: Download All Artifacts Locally (ELI5: Pack all homework into a neat zip bag to submit)

In [ ]:
# Zips all assets, figures, configs, and json results for submission
!zip -r submission_artifacts.zip assets/ report.pdf configs/

try:
    from google.colab import files
    files.download('submission_artifacts.zip')
    print("Downloaded submission_artifacts.zip!")
except Exception as e:
    print(f"Running outside Google Colab environment. File saved as submission_artifacts.zip in current directory ({e}).")